In [2]:
import pandas as pd
from datetime import datetime
from feast import FeatureStore

### 1. Инициализация Feature Store

In [3]:
store = FeatureStore(repo_path=".")

print(f"Project: {store.config.project}")

Project: feature_repo


### 2. Исторические данные (для обучения)

In [4]:
# 2.1 Создаём entity_df с историческими временными метками
entity_df = pd.DataFrame.from_dict({
    "driver_id": [1001, 1002, 1003],
    "event_timestamp": [
        datetime(2021, 4, 12, 10, 59, 42),
        datetime(2021, 4, 12, 8, 12, 10),
        datetime(2021, 4, 12, 16, 40, 26),
    ],
    # Параметры для On-Demand Feature View
    "current_trip_distance": [15.5, 8.2, 42.1],
})

print("Entity DataFrame для исторического запроса:")
entity_df

Entity DataFrame для исторического запроса:


,driver_id,event_timestamp,current_trip_distance
0,1001,2021-04-12 10:59:42,15.5
1,1002,2021-04-12 08:12:10,8.2
2,1003,2021-04-12 16:40:26,42.1


In [5]:
# 2.2 Запрос исторических признаков через Feature Service
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=store.get_feature_service("driver_fraud_features"),
).to_df()

print("\n Исторические признаки (для обучения):")
training_df.head()


 Исторические признаки (для обучения):


,driver_id,event_timestamp,current_trip_distance,conv_rate,acc_rate,avg_daily_trips
0,1001,2021-04-12 10:59:42+00:00,15.5,0.793407,0.691618,311
1,1002,2021-04-12 08:12:10+00:00,8.2,0.847754,0.246087,294
2,1003,2021-04-12 16:40:26+00:00,42.1,0.062295,0.435699,935


In [6]:
# 2.3 Запрос исторических признаков с On-Demand Feature View
training_df_with_risk = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_transaction_stats:conv_rate",
        "driver_transaction_stats:acc_rate",
        "driver_behavior_features:avg_daily_trips",
        "driver_risk_metrics:risk_score",
        "driver_risk_metrics:trip_anomaly",
    ],
).to_df()

print("\n Исторические признаки с On-Demand метриками:")
training_df_with_risk.head()


 Исторические признаки с On-Demand метриками:


,driver_id,event_timestamp,current_trip_distance,conv_rate,acc_rate,avg_daily_trips,risk_score,trip_anomaly
0,1001,2021-04-12 10:59:42+00:00,15.5,0.793407,0.691618,311,0.752692,0.049839
1,1002,2021-04-12 08:12:10+00:00,8.2,0.847754,0.246087,294,0.607087,0.027891
2,1003,2021-04-12 16:40:26+00:00,42.1,0.062295,0.435699,935,0.211657,0.045027


### 3. Онлайн-данные (для инференса)

In [7]:
# 3.1 Сначала материализуем признаки в онлайн-хранилище
from datetime import datetime
from feast import FeatureStore
# Инициализация
store = FeatureStore(repo_path=".")

# Материализуем все данные
store.materialize_incremental(end_date=datetime.now())

Materializing 2 feature views to 2026-09-06 18:32:38+00:00 into the sqlite online store.

driver_behavior_features from 2026-08-30 13:32:38+00:00 to 2026-09-06 18:32:38+00:00:
driver_transaction_stats from 2026-09-05 13:32:38+00:00 to 2026-09-06 18:32:38+00:00:


In [8]:
# 3.2 Онлайн-запрос через Feature Service
online_features = store.get_online_features(
    features=store.get_feature_service("driver_fraud_features"),
    entity_rows=[
        {"driver_id": 1001},
        {"driver_id": 1002},
        {"driver_id": 1003},
    ],
).to_dict()

print("\n Онлайн-признаки:")
for key, value in online_features.items():
    print(f"{key}: {value}")


 Онлайн-признаки:
driver_id: [1001, 1002, 1003]
acc_rate: [1.0, 0.9154895544052124, 0.23925286531448364]
conv_rate: [1.0, 0.8379756808280945, 0.3682475984096527]
avg_daily_trips: [1000, 182, 113]


In [9]:
# 3.3 Онлайн-запрос с On-Demand
online_features_with_risk = store.get_online_features(
    features=[
        "driver_transaction_stats:conv_rate",
        "driver_transaction_stats:acc_rate",
        "driver_behavior_features:avg_daily_trips",
        "driver_risk_metrics:risk_score",
        "driver_risk_metrics:trip_anomaly",
    ],
    entity_rows=[
        {"driver_id": 1001, "current_trip_distance": 15.5},
        {"driver_id": 1002, "current_trip_distance": 8.2},
        {"driver_id": 1003, "current_trip_distance": 42.1},
    ],
).to_dict()

print("\n Онлайн-признаки с On-Demand:")
for key, value in online_features_with_risk.items():
    print(f"{key}: {value}")


 Онлайн-признаки с On-Demand:
driver_id: [1001, 1002, 1003]
acc_rate: [1.0, 0.9154895544052124, 0.23925286531448364]
conv_rate: [1.0, 0.8379756808280945, 0.3682475984096527]
avg_daily_trips: [1000, 182, 113]
risk_score: [1.0, 0.8689812302589417, 0.3166497051715851]
trip_anomaly: [0.015499999845000003, 0.04505494257939875, 0.37256633871094347]
